# 第 1 周结束练习 —— 技术问答解释器

## 练习目标（理念）

为展示你对 **OpenAI API** 与本地 **Ollama** 的熟悉程度，请做一个小工具：

- **输入**：一个技术问题（例如「学习 LLM 对做 AI 工作是否必要？」）
- **输出**：清晰、适合初学者理解的解释（不要写太长）
- **额外要求**：用**流式（streaming）**一边生成一边刷新 Markdown，而不是等整段答完才显示

这是你在课程期间自己也能天天用的小助手。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `openai.chat.completions.create(...)` |
| `messages`（system / user） | system 定「怎么答」，user 放具体问题 |
| 流式输出 `stream=True` | 逐块拼接，并用 `update_display` 刷新 |
| OpenAI 云端模型 | `gpt-4o-mini`（常量 `MODEL_GPT`） |
| Ollama 本地模型 | `llama3.2:1b`（常量 `MODEL_LLAMA`），走 OpenAI 兼容 `/v1` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `OPENAI_API_KEY`；若要用 Llama，请先启动 Ollama 并拉取 `llama3.2:1b`
3. 在「提问」单元格改写 `question`，再分别跑 GPT 与 Llama 两格，对比回答风格


In [7]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 导入标准库 json：本练习导入了但后面未使用，保留以不改逻辑
import json
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染、display 首次显示、update_display 流式刷新
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端类：既可调云端，也可指向本地 Ollama 的兼容接口
from openai import OpenAI


In [8]:
# ========== 环境：加载 .env 并做一次 API Key 形态检查 ==========

# 加载 .env；override=True 表示用文件里的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 从环境变量读取 OpenAI 密钥（名字必须是 OPENAI_API_KEY，和官方 SDK 默认一致）
api_key = os.getenv('OPENAI_API_KEY')

# 粗检：有值、以 sk-proj- 开头、长度够长——仅作学习期提示，不是严格校验
if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    # 成功提示文案保持英文（原样），避免改行为相关输出
    print('API key looks good')
else:
    print('There is a problem finding API key.')


API key looks good


In [18]:
# ========== 常量与客户端：云端 GPT + 本地 Ollama（OpenAI 兼容） ==========

# OpenAI 云端小模型：便宜、够用，适合做解释类问答
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull；字符串必须和本机已安装的模型名一致
MODEL_LLAMA = 'llama3.2:1b'

# 默认 OpenAI 客户端：会从环境变量 OPENAI_API_KEY 自动取密钥
openai = OpenAI()

# Ollama 的 OpenAI 兼容基址（/v1），本机默认端口 11434
OLLAMA_BASE_URL = "http://127.0.0.1:11434/v1"
# 再造一个 OpenAI 客户端，但 base_url 指向本地；api_key 对 Ollama 通常任意非空即可
ollama = OpenAI(base_url = OLLAMA_BASE_URL, api_key="ollama")


In [5]:
# ========== 系统提示（system prompt）：规定模型「怎么答」 ==========

# 发给模型的指令字符串保持英文原样：翻译会改变回答风格/行为
system_prompt = """ You are an expert in answering questions.
Answer the questions asked in a explainbale way for a biggener to understand.
Do not make it too lengthy.
"""


In [21]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 用户问题（user prompt）保持英文原样；练习时可换成你自己的技术问题
question = """
Answer the below question.
Is learning LLM is necessary for working with AI?}
"""


In [22]:
# ========== 路径 A：用云端 gpt-4o-mini 流式回答 ==========

# 定义流式问答函数：system + user → Chat Completions，边收边刷新 Markdown
def stream_answer(system_prompt, question):
    # 调用 OpenAI Chat Completions；stream=True 表示持续返回增量 delta
    stream = openai.chat.completions.create(
        model = MODEL_GPT,
        messages = [
            # system：角色与答题风格
            {"role":"system", "content":system_prompt},
            # user：真正的问题
            {"role":"user", "content":question}
        ],
        stream=True
    )
    # response：把每一块文本拼成完整答案，供 Markdown 展示
    response = ""
    # 先放一个空的 Markdown 占位，拿到 display_id，后面才能原地更新
    display_handle = display(Markdown(""), display_id=True)
    # 遍历流式事件：每个 chunk 可能带一小段 delta.content
    for chunk in stream:
        # or ''：有的 chunk 没有 content（例如结束标记），用空串避免 TypeError
        response += chunk.choices[0].delta.content or ''
        # 用同一 display_id 刷新，实现「打字机」效果
        update_display(Markdown(response), display_id=display_handle.display_id)

# 立刻用上面的 system_prompt 与 question 跑一遍 GPT
stream_answer(system_prompt, question)


Learning about Large Language Models (LLMs) can be very beneficial for working with AI, but it is not strictly necessary. 

Here's why:

1. **Understanding Concepts**: Knowing LLMs helps you understand how AI processes and generates language, which is essential for many AI applications like chatbots or language translation.

2. **Career Opportunities**: Many AI jobs involve LLMs, so having this knowledge can make you more competitive in the job market.

3. **Hands-On Projects**: LLMs are widely used in practical projects. Familiarity with them allows you to work on interesting and relevant tasks.

However, AI is a broad field, and there are many areas to explore, such as computer vision, robotics, or data analysis, that may not require in-depth knowledge of LLMs. It ultimately depends on your specific interests and career goals in AI.

In [23]:
# ========== 路径 B：用本地 Llama 3.2（Ollama）流式回答 ==========

# 函数名与结构刻意和上一格相同，方便对比「只换客户端/模型」即可切换后端
def stream_answer(system_prompt, question):
    # 走 ollama 客户端（base_url 指向本地 /v1），模型名为 MODEL_LLAMA
    stream = ollama.chat.completions.create(
        model = MODEL_LLAMA,
        messages = [
            {"role":"system", "content":system_prompt},
            {"role":"user", "content":question}
        ],
        stream=True
    )
    # 同样先拼完整文本，再配合 update_display 刷新
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

# 用同一问题问本地 Llama，对比风格与速度
stream_answer(system_prompt, question)


**Learning Large Language Models (LLM)**

No, learning Large Language Models (LLM) is more like **using them**, rather than studying one. 

Imagine you're a restaurant owner. You don't need to learn how to cook every single dish in your menu. That's what AI engineers do with LLMs - they use them as a tool to understand and generate text, rather than learning how to perform specific tasks like cooking.

However, having understanding of the subject matter, data, and concepts behind LLMs can be beneficial for working with these models effectively.